# ORACLE: TCIP Molecule Design (TCD Module)

Design Transcriptional Chemical Inducers of Proximity (TCIPs) — bifunctional small molecules
that recruit epigenetic writers/erasers to TF binding sites to enforce reversion perturbations.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

DATA_DIR = Path('../data')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Define Switch Set (from RSP)

In [ ]:
# Example switch set from RSP analysis
# In practice this comes from switch_optimizer.optimize()
switch_perturbations = {
    'CDX2': 'Activation',   # Activate CDX2 (colorectal differentiation)
    'SNAI2': 'Repression',  # Repress SNAI2 (EMT driver)
}
print('Switch perturbations:', switch_perturbations)

## 2. TF Structure Preparation

In [ ]:
from oracle.tcd.tf_structurer import TFStructurer, TCDConfig

tcd_config = TCDConfig()
structurer = TFStructurer(tcd_config)

tf_structures = {}
for tf, ptype in switch_perturbations.items():
    print(f'Preparing structure for {tf} ({ptype})...')
    try:
        result = structurer.prepare(tf, ptype)
        tf_structures[tf] = result
        print(f'  Structure: {result.structure is not None}')
        print(f'  Binding site: {result.binding_site is not None}')
        print(f'  Domains: {len(result.domains)}')
    except Exception as e:
        print(f'  Warning: {e} (using stub)')
        tf_structures[tf] = None

## 3. Writer/Eraser Selection

In [ ]:
from oracle.tcd.writer_selector import WriterEraserSelector

selector = WriterEraserSelector()

selections = {}
for tf, ptype in switch_perturbations.items():
    sel = selector.select(tf, ptype, cancer_expression={}, encode_data={})
    selections[tf] = sel
    print(f'{tf} ({ptype}): recruit {sel.writer_eraser_name} via {sel.recruiter_scaffold}')
    print(f'  Mechanism: {sel.mechanism}')
    print(f'  Score: {sel.selection_score:.4f}')
    print(f'  Recruiter SMILES: {sel.recruiter_smiles[:60]}...')

## 4. Molecule Generation (DDPM Diffusion)

In [ ]:
from oracle.tcd.molecule_generator import MoleculeGenerator

generator = MoleculeGenerator(tcd_config)

# Try loading checkpoint if available
checkpoint_path = Path('../checkpoints/tcd_diffusion.pt')
if checkpoint_path.exists():
    generator.load_model(str(checkpoint_path))
    print('Loaded diffusion model checkpoint')
else:
    print('No checkpoint found — using untrained model for demonstration')

generated_warheads = {}
for tf, ptype in switch_perturbations.items():
    print(f'\nGenerating warheads for {tf}...')
    try:
        warheads = generator.sample(
            pocket_graph=None,
            recruiter_graph=None,
            geometry_constraint=torch.zeros(6),
            n_atoms=25,
            n_samples=5
        )
        generated_warheads[tf] = warheads
        print(f'  Generated {len(warheads)} warhead candidates')
        for i, mol in enumerate(warheads[:3]):
            print(f'  Mol {i}: SMILES={mol.smiles[:40]}... Ki={mol.predicted_ki:.4f}')
    except Exception as e:
        print(f'  Warning: {e}')

## 5. Linker Design

In [ ]:
from oracle.tcd.linker_designer import LinkerDesigner

linker_designer = LinkerDesigner(tcd_config)

for tf in switch_perturbations.keys():
    linker_design = linker_designer.design(
        tf_warhead_pose=None,
        recruiter_warhead_pose=None,
        tf_structure=tf_structures.get(tf),
        writer_structure=None
    )
    print(f'{tf} linker: {linker_design.linker_name}')
    print(f'  SMILES: {linker_design.linker_smiles}')
    print(f'  Required distance: {linker_design.required_distance:.1f} Å')
    print(f'  Linker length: {linker_design.linker_length:.1f} Å')
    print(f'  Flexibility: {linker_design.flexibility}')

## 6. Ternary Complex Validation

In [ ]:
from oracle.tcd.ternary_validator import TernaryComplexValidator

validator = TernaryComplexValidator(tcd_config)

# Validate a candidate TCIP
from oracle.tcd.molecule_generator import Molecule
example_tcip = Molecule(smiles='CC1=CC=C(C=C1)NC(=O)C2=CC=CC=C2', predicted_ki=0.1)

validation = validator.validate(
    tcip_molecule=example_tcip,
    tf_structure=tf_structures.get('CDX2'),
    writer_structure=None,
    tf_warhead_pose=None,
    recruiter_pose=None
)
print(f'Validation result: passed={validation.passed}')
print(f'  Clash score: {validation.clash_score:.2f} (max={tcd_config.max_clash_score})')
print(f'  Interface energy: {validation.interface_energy:.2f} (max={tcd_config.max_interface_energy})')
print(f'  SA score: {validation.sa_score:.2f} (max={tcd_config.max_sa_score})')
print(f'  Drug-like: {validation.drug_like}')

## 7. Molecule Visualization

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Draw, Descriptors
    from IPython.display import Image

    # Visualize writer/eraser scaffolds
    scaffolds = {
        'BRD4 recruiter (JQ1 derivative)': 'CC1=C(SC2=CC=CC=C12)C3=CC=C(C=C3)CC4=NN=C5SC=CC5=N4',
        'EZH2 recruiter (EPZ-6438)': 'CCN1CCN(CC1)CC2=CC=C(C=C2)NC3=NC(=O)C4=CC=CC=C4N3'
    }
    mols = []
    legends = []
    for name, smiles in scaffolds.items():
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            mols.append(mol)
            mw = Descriptors.MolWt(mol)
            legends.append(f'{name}\nMW={mw:.1f}')
    
    if mols:
        img = Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(400, 300),
                                    legends=legends)
        display(img)
except ImportError:
    print('RDKit not available for visualization')